In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import langchain # Add this import
# 1. Initialize the Wikipedia Tool
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1500)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)
tools = [wiki_tool]

# 2. Initialize Local Llama 3.1 
llm = ChatOllama(
    model="llama3.1",
    temperature=0 
)


# 3. THE MAIN DIFFERENCE: Bind the tools to the LLM
# Ye LLM ko batata hai ki uske paas tools available hain
llm_with_tools = llm.bind_tools(tools)

# 4. Create a simple Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools if necessary."),
    ("human", "{input}")
])

# 5. Create the Chain (LCEL - LangChain Expression Language)
chain = prompt | llm_with_tools

# 6. Test it!
print("Asking Llama 3.1 a question...")
response = chain.invoke({
    "input": "What is the capital of Bihar and what is it famous for?"
})

# 7. Check the output
print("\n--- LLM RESPONSE ---")
# Agar LLM ko lagega ki tool use karna hai, toh wo text ki jagah 'tool_calls' dega
if response.tool_calls:
    print("Llama 3.1 decided to use a tool!")
    print(f"Tool to call: {response.tool_calls[0]['name']}")
    print(f"Arguments: {response.tool_calls[0]['args']}")
else:
    print(response.content)

Asking Llama 3.1 a question...

--- LLM RESPONSE ---
Llama 3.1 decided to use a tool!
Tool to call: wikipedia
Arguments: {'query': 'Capital of Bihar and its notable features'}
